# Litigi per-user analysis

Example: https://scikit-learn.org/stable/auto_examples/text/plot_document_clustering.html#sphx-glr-auto-examples-text-plot-document-clustering-py

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from functions.utilities import preprocess
from stop_words import get_stop_words
from time import time
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import PCA

In [ ]:
stop = get_stop_words('italian')


In [ ]:
filename='processed_data\itigi_comments_copnsolidated.parqet'
litigi=pd.read_parquet(filename).drop_duplicates(subset=['author','body','id', 'created_utc'])
litigi['created_dt']=(pd.to_datetime(litigi['created_utc'],unit='s'))
litigi['body_preprocessed']=litigi['body'].apply(lambda x: preprocess(x))

# extract the author of the parent comment (if applicable)
id_to_author = litigi.set_index('id')['author'].to_dict()
litigi['parent_author'] = litigi['parent_id'].str.split('_').str[1].map(id_to_author)

litigi.head()

In [ ]:
litigi['created_hour']=litigi['created_dt'].dt.hour
litigi

In [ ]:
first_n_comments=50000

grouped_by_user=litigi[litigi['author']!='[deleted]'][:first_n_comments].groupby('author')


In [ ]:
hour=litigi[litigi['author']!='[deleted]'][:first_n_comments].groupby(['author','created_hour']).count()
user_comments_binned=hour[['body']].reset_index().pivot_table(index='author',columns='created_hour',aggfunc='sum').reset_index().fillna(0)
user_comments_binned

In [ ]:
user_comments=grouped_by_user['body_preprocessed'].agg(lambda x: ' '.join(x)).reset_index()
user_comments

In [ ]:
t0 = time()
vectorizer = TfidfVectorizer(stop_words=stop,max_features=200)
X_tfidf = vectorizer.fit_transform(user_comments['body_preprocessed'])
print(f"vectorization done in {time() - t0:.3f} s")
print(f"n_samples: {X_tfidf.shape[0]}, n_features: {X_tfidf.shape[1]}")

In [ ]:
(vectorizer.get_feature_names_out())

In [ ]:
labels=user_comments['author']

In [ ]:



lsa = make_pipeline(TruncatedSVD(n_components=100), Normalizer(copy=False))
t0 = time()
X_lsa = lsa.fit_transform(X_tfidf)
explained_variance = lsa[0].explained_variance_ratio_.sum()

print(f"LSA done in {time() - t0:.3f} s")
print(f"Explained variance of the SVD step: {explained_variance * 100:.1f}%")

In [ ]:
pca=make_pipeline(lsa,PCA(n_components=2))
X_pca = pca.fit_transform(X_tfidf)
X_pca

In [ ]:
px.scatter(x=X_pca[:,0],y=X_pca[:,1],labels=labels,color=labels)

In [ ]:
user_comments

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np
# Vectorize text
vectorizer = CountVectorizer(stop_words=stop)
X = vectorizer.fit_transform(litigi[:10000]['body_preprocessed'])

# Apply LDA
lda = LatentDirichletAllocation(n_components=10, random_state=42)
lda.fit(X)

# Display topics
topics = np.argsort(lda.components_, axis=1)[:, -5:]
print("Top words per topic:")
for i, topic in enumerate(topics):
    print(f"Topic {i}: {[vectorizer.get_feature_names_out()[j] for j in topic]}")

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wordcloud = WordCloud(width=1600, height=800,stopwords=stop).generate(" ".join(user_comments['body_preprocessed']))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()